# Lab 1 — Sampling, CLT và Bootstrap

**Thời lượng:** 120–150 phút  
**Mục tiêu:** phân biệt data/sampling distribution, kiểm chứng standard error và tự xây percentile bootstrap CI.

> Mọi experiment phải ghi sample size, số trials/resamples và seed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
OUTPUT = ROOT / 'outputs'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print('Project root:', ROOT.resolve())

## 1. Population distribution và descriptive statistics

Trong lab mô phỏng này, CSV 5.000 rows đóng vai finite population nên ta biết population parameter để đối chiếu sampling error.

In [ ]:
population = pd.read_csv(ROOT / 'data' / 'skewed_population.csv')
spend = population['monthly_spend'].to_numpy(dtype=float)
population_summary = pd.Series({
    'rows': len(spend),
    'mean': spend.mean(),
    'median': np.median(spend),
    'variance_ddof0': np.var(spend, ddof=0),
    'std_ddof0': np.std(spend, ddof=0),
    'q25': np.quantile(spend, 0.25),
    'q75': np.quantile(spend, 0.75),
    'q95': np.quantile(spend, 0.95),
})
print(population_summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(spend, bins=50, color='steelblue', alpha=0.85)
axes[0].axvline(spend.mean(), color='crimson', label='Mean')
axes[0].axvline(np.median(spend), color='black', linestyle='--', label='Median')
axes[0].set(title='Raw monthly_spend', xlabel='Spend', ylabel='Count')
axes[0].legend()
axes[1].hist(np.log1p(spend), bins=50, color='darkorange', alpha=0.85)
axes[1].set(title='log1p(monthly_spend)', xlabel='log1p(spend)', ylabel='Count')
fig.tight_layout()
fig.savefig(FIGURES / 'population_distribution.png', dpi=150)
plt.show()

## 2. Conditional probability bằng event masks

Luôn đọc điều kiện sau dấu `|` làm denominator.

In [ ]:
premium = population['segment'].eq('premium').to_numpy()
converted = population['converted'].eq(1).to_numpy()
p_convert_given_premium = (converted & premium).sum() / premium.sum()
p_premium_given_convert = (converted & premium).sum() / converted.sum()
contingency = pd.crosstab(
    population['segment'], population['converted'], margins=True, normalize=False
)
print(contingency)
print(f'P(converted | premium) = {p_convert_given_premium:.6f}')
print(f'P(premium | converted) = {p_premium_given_convert:.6f}')
assert not np.isclose(p_convert_given_premium, p_premium_given_convert)

## 3. Sampling distribution của mean

Ta lặp 2.000 trials cho từng sample size. Sampling có replacement giúp mô phỏng các sample IID từ empirical population.

In [ ]:
def simulate_sample_means(values, sample_size, n_trials, rng):
    indices = rng.integers(0, len(values), size=(n_trials, sample_size))
    return values[indices].mean(axis=1)


sample_sizes = [5, 20, 100]
sampling_distributions = {}
rng = np.random.default_rng(20260301)
for sample_size in sample_sizes:
    sampling_distributions[sample_size] = simulate_sample_means(
        spend, sample_size, 2_000, rng
    )

In [ ]:
sampling_rows = []
population_std = np.std(spend, ddof=0)
for sample_size, means in sampling_distributions.items():
    sampling_rows.append({
        'sample_size': sample_size,
        'mean_of_sample_means': means.mean(),
        'empirical_standard_error': means.std(ddof=1),
        'theoretical_standard_error': population_std / np.sqrt(sample_size),
        'absolute_bias': abs(means.mean() - spend.mean()),
    })
sampling_summary = pd.DataFrame(sampling_rows)
sampling_summary.to_csv(OUTPUT / 'sampling_summary.csv', index=False)
sampling_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
for ax, sample_size in zip(axes, sample_sizes):
    means = sampling_distributions[sample_size]
    ax.hist(means, bins=35, alpha=0.85)
    ax.axvline(spend.mean(), color='crimson', linestyle='--', label='Population mean')
    ax.set(title=f'n={sample_size}', xlabel='Sample mean', ylabel='Trials')
axes[0].legend()
fig.suptitle('Sampling distributions of the mean')
fig.tight_layout()
fig.savefig(FIGURES / 'sampling_distributions.png', dpi=150)
plt.show()

## 4. Kiểm tra `SE ≈ σ/√n`

Empirical/theoretical ratio gần 1 là bằng chứng mô phỏng. Không yêu cầu đúng tuyệt đối vì chỉ có 2.000 trials.

In [ ]:
sampling_summary['se_ratio'] = (
    sampling_summary['empirical_standard_error']
    / sampling_summary['theoretical_standard_error']
)
print(sampling_summary[['sample_size', 'se_ratio']].to_string(index=False))
assert sampling_summary['se_ratio'].between(0.90, 1.10).all()
assert sampling_summary.loc[2, 'empirical_standard_error'] < sampling_summary.loc[0, 'empirical_standard_error']

## 5. Percentile bootstrap CI từ đầu

Lấy một sample 120 rows **không replacement** từ population. Sau đó bootstrap từ sample này **có replacement**.

In [ ]:
sample_rng = np.random.default_rng(314)
sample_indices = sample_rng.choice(len(spend), size=120, replace=False)
observed_sample = spend[sample_indices]
print('Sample mean:', observed_sample.mean())
print('Sample median:', np.median(observed_sample))

In [ ]:
def percentile_bootstrap(values, statistic, n_resamples, confidence_level, random_state):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(random_state)
    distribution = np.empty(n_resamples, dtype=float)
    for b in range(n_resamples):
        indices = rng.integers(0, len(values), size=len(values))
        distribution[b] = statistic(values[indices])
    alpha = (1.0 - confidence_level) / 2.0
    lower, upper = np.quantile(distribution, [alpha, 1.0 - alpha])
    return float(statistic(values)), float(lower), float(upper), distribution

In [ ]:
mean_est, mean_low, mean_high, mean_boot = percentile_bootstrap(
    observed_sample, np.mean, 4_000, 0.95, 2718
)
median_est, median_low, median_high, median_boot = percentile_bootstrap(
    observed_sample, np.median, 4_000, 0.95, 2718
)
bootstrap_intervals = pd.DataFrame([
    {'statistic': 'mean', 'estimate': mean_est, 'lower_95': mean_low, 'upper_95': mean_high},
    {'statistic': 'median', 'estimate': median_est, 'lower_95': median_low, 'upper_95': median_high},
])
bootstrap_intervals['width'] = bootstrap_intervals['upper_95'] - bootstrap_intervals['lower_95']
bootstrap_intervals.to_csv(OUTPUT / 'bootstrap_intervals.csv', index=False)
bootstrap_intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, distribution, estimate, low, high, name in [
    (axes[0], mean_boot, mean_est, mean_low, mean_high, 'Mean'),
    (axes[1], median_boot, median_est, median_low, median_high, 'Median'),
]:
    ax.hist(distribution, bins=40, alpha=0.85)
    ax.axvline(estimate, color='black', label='Estimate')
    ax.axvspan(low, high, color='orange', alpha=0.25, label='95% percentile CI')
    ax.set(title=f'Bootstrap distribution — {name}', xlabel=name, ylabel='Resamples')
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'bootstrap_distribution.png', dpi=150)
plt.show()

## 6. Diễn giải và limitation

Diễn giải đúng: nếu lặp lại sampling từ population và cùng quy trình interval nhiều lần, khoảng 95% intervals sẽ chứa parameter dưới các giả định. Interval này không xử lý selection bias, data drift hoặc dependence giữa rows.

**Exit ticket:** Vì sao bootstrap dùng replacement? Vì sao tăng `n` làm sampling distribution của mean hẹp hơn?

In [ ]:
assert abs(spend.mean() - 40.69777946) < 1e-8
assert abs(p_convert_given_premium - 0.3770491803278688) < 1e-12
assert abs(mean_est - 41.47249583333333) < 1e-10
assert abs(mean_low - 37.13284254166666) < 1e-10
assert abs(mean_high - 46.05902839583334) < 1e-10
assert (FIGURES / 'sampling_distributions.png').exists()
print('PASS — sampling, CLT, conditional probability và bootstrap checks đạt.')